In [0]:
%run ./utils

#Import

In [0]:
from pyspark.sql import functions as F

In [0]:
df_seg = spark.range(24 * 60 * 60).select(F.col("id").cast("int").alias("horario"))

# Transformações

In [0]:
# 1 linha por segundo; hora, minuto e segundo saem de divisão inteira
df_seg = spark.range(24 * 60 * 60).select(F.col("id").cast("int").alias("horario"))

df_dim = (
    df_seg
    .withColumn("hora",    F.expr("horario DIV 3600").cast("int"))
    .withColumn("minuto",  F.expr("(horario % 3600) DIV 60").cast("int"))
    .withColumn("segundo", F.expr("horario % 60").cast("int"))
    .withColumn("horario_sk", (F.col("hora") * 10000 + F.col("minuto") * 100 + F.col("segundo")).cast("bigint"))
    .withColumn("horario_txt", F.format_string("%02d:%02d:%02d", F.col("hora"), F.col("minuto"), F.col("segundo")))
    .withColumn(
        "periodo",
        F.when(F.col("hora").between(0,7), "Madrugada")
         .when(F.col("hora").between(8,12), "Manhã")
         .when(F.col("hora").between(3,18), "Tarde")
         .otherwise("Noite")
    )
    .withColumn(
        "horario_comercial",
        (F.col("hora") >= 8) & (F.col("hora") < 18)
    )
    .select("horario_sk", "horario", "hora", "minuto", "segundo", "horario_txt", "periodo", "horario_comercial")
)

In [0]:
df_unknown = spark.range(1).select(
    F.lit(-1).cast("bigint").alias("horario_sk"),
    F.lit(None).cast("int").alias("horario"),
    F.lit(None).cast("int").alias("hora"),
    F.lit(None).cast("int").alias("minuto"),
    F.lit(None).cast("int").alias("segundo"),
    F.lit("NÃO INFORMADO").alias("horario_txt"),
    F.lit("NÃO INFORMADO").alias("periodo"),
    F.lit(False).alias("horario_comercial"),
)

df_dim = df_dim.unionByName(df_unknown)

# Salvamento

In [0]:
dbutils.widgets.text("gold_table_name", "dim_horario")
gold_table_name = dbutils.widgets.get("gold_table_name")

save_df(df_dim, gold_table_name)

In [0]:
display(df_dim.orderBy("horario_sk").limit(10))